# GVH Diagonal Cubic 0.2.23.5.1 — Timelike Field Physical Postulate and Origin — Traceability Fix

**Auteur : Charlemagne O Laurince**

---

## Objectif

Le notebook `0.2.23.4` a montré qu’aucun des trois modèles candidats de \(u^\mu\) n’est sélectionné par les postulats GVH actuels.

Le présent notebook revient donc aux objets déjà présents dans GVH Diagonal Cubic et pose la question :

\[
\boxed{
u^\mu\ \text{peut-il être dérivé d’une structure GVH existante ?}
}
\]

Les origines candidates étudiées sont :

1. direction temporelle propre de l’observateur ;
2. direction temporelle issue d’un champ scalaire d’horloge ;
3. direction propre principale d’un tenseur GVH ;
4. quadrivitesse de la matière ;
5. champ vectoriel indépendant ajouté par postulat.

---

## Critère central

Une origine physique est considérée comme dérivée seulement si elle fournit :

\[
u^\mu u_\mu=-1,
\]

une définition covariante,

une direction unique dans le vide et dans la matière,

une limite GR contrôlée,

et au moins une conséquence falsifiable.

Une construction simplement possible n’est pas une dérivation.

---

## Statuts possibles

```text
PASS-TIMELIKE-FIELD-DERIVED-FROM-GVH-GEOMETRY
PASS-CANDIDATE-TIMELIKE-ORIGINS-IDENTIFIED
BLOCKED-UNIQUE-PHYSICAL-ORIGIN
NEW-PHYSICAL-POSTULATE-REQUIRED
```

---

## Correction 0.2.23.5.1

Cette version met à jour la dépendance canonique :

```text
0.2.23.3
```

vers :

```text
0.2.23.3.1
```

La logique scientifique, les calculs et le verdict restent inchangés.


In [1]:
from __future__ import annotations

import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sympy as sp

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_colwidth", 240)

print("Python :", sys.version)
print("SymPy :", sp.__version__)

Python : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
SymPy : 1.14.0


# 1. Dépôt et chemins

In [2]:
REPOSITORY_URL = "https://github.com/col38470682/Univers.git"
REPOSITORY_DIR = Path("/content/Univers")
PROJECT_ROOT = REPOSITORY_DIR / "gvh_diagonal_cubic"

if (REPOSITORY_DIR / ".git").exists():
    subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", REPOSITORY_URL, str(REPOSITORY_DIR)],
        check=True,
    )

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(PROJECT_ROOT)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "timelike_field"
)

EXPORT_DIR = PROJECT_ROOT / "exports"

for directory in [PROCESSED_DIR, EXPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)

PROJECT_ROOT : /content/Univers/gvh_diagonal_cubic


# 2. Dépendances théoriques

In [3]:
CANONICAL_VALIDATION_DIR = (
    PROJECT_ROOT
    / "0.2_gravity"
    / "0.2B_astrophysics_validation"
)

NOTEBOOK_SPECS = {
    "0.2.21": {
        "canonical_name": (
            "GVH_Diagonal_Cubic_0.2.21_"
            "Equivalence_Principle_PPN_Constraints.ipynb"
        ),
        "fallback_pattern": (
            "*0.2.21*Equivalence*Principle*PPN*Constraints*.ipynb"
        ),
    },
    "0.2.22": {
        "canonical_name": (
            "GVH_Diagonal_Cubic_0.2.22_"
            "Static_Spherical_Solution_PPN_Derivation.ipynb"
        ),
        "fallback_pattern": (
            "*0.2.22*Static*Spherical*PPN*Derivation*.ipynb"
        ),
    },
    "0.2.23": {
        "canonical_name": (
            "GVH_Diagonal_Cubic_0.2.23_"
            "Covariant_Static_Field_Equations_Source_Matching.ipynb"
        ),
        "fallback_pattern": (
            "*0.2.23*Covariant*Static*Field*Source*Matching*.ipynb"
        ),
    },
    "0.2.23.1": {
        "canonical_name": (
            "GVH_Diagonal_Cubic_0.2.23.1_"
            "Weak_Field_Coefficient_Extraction_for_PPN.ipynb"
        ),
        "fallback_pattern": (
            "*0.2.23.1*Weak*Field*Coefficient*Extraction*PPN*.ipynb"
        ),
    },
    "0.2.23.2": {
        "canonical_name": (
            "GVH_Diagonal_Cubic_0.2.23.2_"
            "Covariant_Source_Projector_Derivation.ipynb"
        ),
        "fallback_pattern": (
            "*0.2.23.2*Covariant*Source*Projector*Derivation*.ipynb"
        ),
    },
    "0.2.23.3.1": {
        "canonical_name": (
            "GVH_Diagonal_Cubic_0.2.23.3.1_"
            "Timelike_Field_Dynamics_and_Covariant_Closure_"
            "Dependency_Detection_Fix.ipynb"
        ),
        "fallback_pattern": (
            "*0.2.23.3.1*Timelike*Field*Dynamics*Covariant*Closure*.ipynb"
        ),
    },
    "0.2.23.4.1": {
        "canonical_name": (
            "GVH_Diagonal_Cubic_0.2.23.4.1_"
            "Timelike_Field_Model_Selection_and_Falsifiability_"
            "Traceability_Fix.ipynb"
        ),
        "fallback_pattern": (
            "*0.2.23.4.1*Timelike*Field*Model*Selection*"
            "Falsifiability*.ipynb"
        ),
    },
}


def resolve_source_notebook(spec):
    canonical_path = (
        CANONICAL_VALIDATION_DIR
        / spec["canonical_name"]
    )

    if canonical_path.exists():
        return canonical_path, "CANONICAL_EXACT_PATH"

    candidates = sorted(
        path
        for path in PROJECT_ROOT.rglob(
            spec["fallback_pattern"]
        )
        if path.is_file()
        and ".ipynb_checkpoints" not in path.parts
    )

    if candidates:
        preferred = [
            path
            for path in candidates
            if path.parent == CANONICAL_VALIDATION_DIR
        ]

        selected = (
            preferred[0]
            if preferred
            else candidates[0]
        )

        return selected, "FALLBACK_RECURSIVE_SEARCH"

    return None, "NOT_FOUND"


resolved_notebooks = {}
resolution_methods = {}

for notebook_id, spec in NOTEBOOK_SPECS.items():
    resolved_path, method = resolve_source_notebook(spec)
    resolved_notebooks[notebook_id] = resolved_path
    resolution_methods[notebook_id] = method

source_notebooks_df = pd.DataFrame([
    {
        "notebook_id": notebook_id,
        "canonical_name": NOTEBOOK_SPECS[
            notebook_id
        ]["canonical_name"],
        "resolved_path": (
            str(path)
            if path is not None
            else ""
        ),
        "resolution_method": resolution_methods[
            notebook_id
        ],
        "found": path is not None,
        "in_canonical_directory": bool(
            path is not None
            and path.parent
            == CANONICAL_VALIDATION_DIR
        ),
    }
    for notebook_id, path
    in resolved_notebooks.items()
])

all_sources_found = bool(
    source_notebooks_df["found"].all()
)

all_sources_canonical = bool(
    source_notebooks_df[
        "in_canonical_directory"
    ].all()
)

source_traceability_pass = bool(
    all_sources_found
    and all_sources_canonical
)

print(
    "Toutes les dépendances trouvées :",
    all_sources_found,
)
print(
    "Toutes dans 0.2B_astrophysics_validation :",
    all_sources_canonical,
)
print(
    "Traçabilité complète :",
    source_traceability_pass,
)

source_notebooks_df

Toutes les dépendances trouvées : True
Toutes dans 0.2B_astrophysics_validation : True
Traçabilité complète : True


,notebook_id,canonical_name,resolved_path,resolution_method,found,in_canonical_directory
0,0.2.21,GVH_Diagonal_Cubic_0.2.21_Equivalence_Principle_PPN_Constraints.ipynb,/content/Univers/gvh_diagonal_cubic/0.2_gravity/0.2B_astrophysics_validation/GVH_Diagonal_Cubic_0.2.21_Equivalence_Principle_PPN_Constraints.ipynb,CANONICAL_EXACT_PATH,True,True
1,0.2.22,GVH_Diagonal_Cubic_0.2.22_Static_Spherical_Solution_PPN_Derivation.ipynb,/content/Univers/gvh_diagonal_cubic/0.2_gravity/0.2B_astrophysics_validation/GVH_Diagonal_Cubic_0.2.22_Static_Spherical_Solution_PPN_Derivation.ipynb,CANONICAL_EXACT_PATH,True,True
2,0.2.23,GVH_Diagonal_Cubic_0.2.23_Covariant_Static_Field_Equations_Source_Matching.ipynb,/content/Univers/gvh_diagonal_cubic/0.2_gravity/0.2B_astrophysics_validation/GVH_Diagonal_Cubic_0.2.23_Covariant_Static_Field_Equations_Source_Matching.ipynb,CANONICAL_EXACT_PATH,True,True
3,0.2.23.1,GVH_Diagonal_Cubic_0.2.23.1_Weak_Field_Coefficient_Extraction_for_PPN.ipynb,/content/Univers/gvh_diagonal_cubic/0.2_gravity/0.2B_astrophysics_validation/GVH_Diagonal_Cubic_0.2.23.1_Weak_Field_Coefficient_Extraction_for_PPN_UPDATED.ipynb,FALLBACK_RECURSIVE_SEARCH,True,True
4,0.2.23.2,GVH_Diagonal_Cubic_0.2.23.2_Covariant_Source_Projector_Derivation.ipynb,/content/Univers/gvh_diagonal_cubic/0.2_gravity/0.2B_astrophysics_validation/GVH_Diagonal_Cubic_0.2.23.2_Covariant_Source_Projector_Derivation.ipynb,CANONICAL_EXACT_PATH,True,True
5,0.2.23.3.1,GVH_Diagonal_Cubic_0.2.23.3.1_Timelike_Field_Dynamics_and_Covariant_Closure_Dependency_Detection_Fix.ipynb,/content/Univers/gvh_diagonal_cubic/0.2_gravity/0.2B_astrophysics_validation/GVH_Diagonal_Cubic_0.2.23.3.1_Timelike_Field_Dynamics_and_Covariant_Closure_Dependency_Detection_Fix.ipynb,CANONICAL_EXACT_PATH,True,True
6,0.2.23.4.1,GVH_Diagonal_Cubic_0.2.23.4.1_Timelike_Field_Model_Selection_and_Falsifiability_Traceability_Fix.ipynb,/content/Univers/gvh_diagonal_cubic/0.2_gravity/0.2B_astrophysics_validation/GVH_Diagonal_Cubic_0.2.23.4.1_Timelike_Field_Model_Selection_and_Falsifiability_Traceability_Fix.ipynb,CANONICAL_EXACT_PATH,True,True


# Audit des dépendances canoniques critiques `0.2.23.3.1` et `0.2.23.4.1`

In [4]:
critical_dependency_ids = [
    "0.2.23.3.1",
    "0.2.23.4.1",
]

critical_dependencies_df = source_notebooks_df.loc[
    source_notebooks_df[
        "notebook_id"
    ].isin(
        critical_dependency_ids
    )
].copy()

critical_dependencies_df[
    "dependency_pass"
] = (
    critical_dependencies_df["found"]
    & critical_dependencies_df[
        "in_canonical_directory"
    ]
)

dependency_02331_pass = bool(
    critical_dependencies_df.loc[
        critical_dependencies_df[
            "notebook_id"
        ] == "0.2.23.3.1",
        "dependency_pass",
    ].all()
)

dependency_02341_pass = bool(
    critical_dependencies_df.loc[
        critical_dependencies_df[
            "notebook_id"
        ] == "0.2.23.4.1",
        "dependency_pass",
    ].all()
)

print(
    "Dépendance 0.2.23.3.1 trouvée et canonique :",
    dependency_02331_pass,
)
print(
    "Dépendance 0.2.23.4.1 trouvée et canonique :",
    dependency_02341_pass,
)

critical_dependencies_df

Dépendance 0.2.23.3.1 trouvée et canonique : True
Dépendance 0.2.23.4.1 trouvée et canonique : True


,notebook_id,canonical_name,resolved_path,resolution_method,found,in_canonical_directory,dependency_pass
5,0.2.23.3.1,GVH_Diagonal_Cubic_0.2.23.3.1_Timelike_Field_Dynamics_and_Covariant_Closure_Dependency_Detection_Fix.ipynb,/content/Univers/gvh_diagonal_cubic/0.2_gravity/0.2B_astrophysics_validation/GVH_Diagonal_Cubic_0.2.23.3.1_Timelike_Field_Dynamics_and_Covariant_Closure_Dependency_Detection_Fix.ipynb,CANONICAL_EXACT_PATH,True,True,True
6,0.2.23.4.1,GVH_Diagonal_Cubic_0.2.23.4.1_Timelike_Field_Model_Selection_and_Falsifiability_Traceability_Fix.ipynb,/content/Univers/gvh_diagonal_cubic/0.2_gravity/0.2B_astrophysics_validation/GVH_Diagonal_Cubic_0.2.23.4.1_Timelike_Field_Model_Selection_and_Falsifiability_Traceability_Fix.ipynb,CANONICAL_EXACT_PATH,True,True,True


# 3. Objets GVH déjà disponibles

In [5]:
existing_objects_df = pd.DataFrame([
    {
        "object": "g_mn",
        "role": "spacetime metric",
        "contains_timelike_direction": "not uniquely",
        "status": "ESTABLISHED",
    },
    {
        "object": "D_mn",
        "role": "directional anisotropic sector",
        "contains_timelike_direction": "unknown",
        "status": "DYNAMICS INCOMPLETE",
    },
    {
        "object": "Pi_mn[T]",
        "role": "spatial STF anisotropic source",
        "contains_timelike_direction": (
            "depends on u^mu but does not define it"
        ),
        "status": "ALGEBRAICALLY DERIVED",
    },
    {
        "object": "tau_G / directional time variables",
        "role": "exploratory temporal geometry",
        "contains_timelike_direction": "candidate only",
        "status": "EXPLORATORY TRACK",
    },
    {
        "object": "matter four-velocity",
        "role": "local material congruence",
        "contains_timelike_direction": "yes in matter",
        "status": "NOT GLOBAL IN VACUUM",
    },
])

existing_objects_df

,object,role,contains_timelike_direction,status
0,g_mn,spacetime metric,not uniquely,ESTABLISHED
1,D_mn,directional anisotropic sector,unknown,DYNAMICS INCOMPLETE
2,Pi_mn[T],spatial STF anisotropic source,depends on u^mu but does not define it,ALGEBRAICALLY DERIVED
3,tau_G / directional time variables,exploratory temporal geometry,candidate only,EXPLORATORY TRACK
4,matter four-velocity,local material congruence,yes in matter,NOT GLOBAL IN VACUUM


# 4. Candidats d’origine physique

In [6]:
origins_df = pd.DataFrame([
    {
        "origin_id": "OBSERVER_CONGRUENCE",
        "definition": "u^mu is the observer congruence",
        "uses_existing_GVH_object": True,
        "new_field_required": False,
    },
    {
        "origin_id": "SCALAR_CLOCK",
        "definition": (
            "u_mu = -grad_mu(phi)/sqrt(-grad(phi)^2)"
        ),
        "uses_existing_GVH_object": False,
        "new_field_required": True,
    },
    {
        "origin_id": "TENSOR_EIGENVECTOR",
        "definition": (
            "u^mu is a normalized timelike eigenvector "
            "of a GVH tensor"
        ),
        "uses_existing_GVH_object": True,
        "new_field_required": False,
    },
    {
        "origin_id": "MATTER_CONGRUENCE",
        "definition": "u^mu is the matter four-velocity",
        "uses_existing_GVH_object": True,
        "new_field_required": False,
    },
    {
        "origin_id": "INDEPENDENT_VECTOR",
        "definition": (
            "u^mu is a new independent unit vector"
        ),
        "uses_existing_GVH_object": False,
        "new_field_required": True,
    },
])

origins_df

,origin_id,definition,uses_existing_GVH_object,new_field_required
0,OBSERVER_CONGRUENCE,u^mu is the observer congruence,True,False
1,SCALAR_CLOCK,u_mu = -grad_mu(phi)/sqrt(-grad(phi)^2),False,True
2,TENSOR_EIGENVECTOR,u^mu is a normalized timelike eigenvector of a GVH tensor,True,False
3,MATTER_CONGRUENCE,u^mu is the matter four-velocity,True,False
4,INDEPENDENT_VECTOR,u^mu is a new independent unit vector,False,True


# 5. Critères de dérivation

In [7]:
criteria_df = pd.DataFrame([
    ("K1", "Covariant definition", True),
    ("K2", "Unit normalization derivable", True),
    ("K3", "Unique timelike direction", True),
    ("K4", "Defined in vacuum", True),
    ("K5", "Defined with multiple matter fluids", False),
    ("K6", "Constructed from existing GVH objects", True),
    ("K7", "No circular dependence on Pi_mn[T]", True),
    ("K8", "GR limit identifiable", True),
    ("K9", "Action or evolution law available", True),
    ("K10", "Falsifiable consequence identifiable", True),
], columns=["criterion_id", "criterion", "hard_gate"])

criteria_df

,criterion_id,criterion,hard_gate
0,K1,Covariant definition,True
1,K2,Unit normalization derivable,True
2,K3,Unique timelike direction,True
3,K4,Defined in vacuum,True
4,K5,Defined with multiple matter fluids,False
5,K6,Constructed from existing GVH objects,True
6,K7,No circular dependence on Pi_mn[T],True
7,K8,GR limit identifiable,True
8,K9,Action or evolution law available,True
9,K10,Falsifiable consequence identifiable,True


# 6. Évaluation des origines candidates

In [8]:
rows = [
    ("OBSERVER_CONGRUENCE","K1",1.0,"Covariant congruence once observer family is specified"),
    ("OBSERVER_CONGRUENCE","K2",1.0,"Can be normalized"),
    ("OBSERVER_CONGRUENCE","K3",0.0,"Observer family is not unique"),
    ("OBSERVER_CONGRUENCE","K4",1.0,"Can be defined in vacuum"),
    ("OBSERVER_CONGRUENCE","K5",1.0,"Independent of fluid multiplicity"),
    ("OBSERVER_CONGRUENCE","K6",0.7,"Observer frame is tracked in GVH but not dynamically selected"),
    ("OBSERVER_CONGRUENCE","K7",1.0,"No circular dependence"),
    ("OBSERVER_CONGRUENCE","K8",1.0,"GR observer congruences exist"),
    ("OBSERVER_CONGRUENCE","K9",0.0,"No GVH evolution law selecting one congruence"),
    ("OBSERVER_CONGRUENCE","K10",0.3,"Predictions are frame-dependent unless a selection rule exists"),

    ("SCALAR_CLOCK","K1",1.0,"Manifestly covariant"),
    ("SCALAR_CLOCK","K2",1.0,"Normalization is automatic"),
    ("SCALAR_CLOCK","K3",1.0,"Unique if grad(phi) is timelike and nonzero"),
    ("SCALAR_CLOCK","K4",1.0,"Can exist in vacuum"),
    ("SCALAR_CLOCK","K5",1.0,"Independent of fluid multiplicity"),
    ("SCALAR_CLOCK","K6",0.0,"Scalar clock is not yet an established GVH field"),
    ("SCALAR_CLOCK","K7",1.0,"No circular dependence"),
    ("SCALAR_CLOCK","K8",0.8,"Requires decoupling or constant-clock limit"),
    ("SCALAR_CLOCK","K9",0.0,"No scalar action derived"),
    ("SCALAR_CLOCK","K10",0.8,"Foliation signatures possible after action is fixed"),

    ("TENSOR_EIGENVECTOR","K1",1.0,"Eigenvector equation is covariant"),
    ("TENSOR_EIGENVECTOR","K2",1.0,"Can be normalized if timelike"),
    ("TENSOR_EIGENVECTOR","K3",0.5,"Uniqueness fails for degenerate eigenvalues"),
    ("TENSOR_EIGENVECTOR","K4",0.5,"Depends on nonzero tensor structure in vacuum"),
    ("TENSOR_EIGENVECTOR","K5",1.0,"Independent of matter-fluid counting"),
    ("TENSOR_EIGENVECTOR","K6",1.0,"Can use D_mn or another GVH tensor"),
    ("TENSOR_EIGENVECTOR","K7",1.0,"Can avoid Pi_mn if based directly on D_mn"),
    ("TENSOR_EIGENVECTOR","K8",0.5,"GR limit may make eigenvector undefined when D_mn vanishes"),
    ("TENSOR_EIGENVECTOR","K9",0.0,"D_mn dynamics not sufficiently closed"),
    ("TENSOR_EIGENVECTOR","K10",0.7,"Eigenframe effects could be falsifiable"),

    ("MATTER_CONGRUENCE","K1",1.0,"Matter four-velocity is covariant"),
    ("MATTER_CONGRUENCE","K2",1.0,"Normalized by definition"),
    ("MATTER_CONGRUENCE","K3",0.5,"Unique only for a single effective fluid"),
    ("MATTER_CONGRUENCE","K4",0.0,"Undefined in vacuum"),
    ("MATTER_CONGRUENCE","K5",0.0,"Ambiguous for non-comoving fluids"),
    ("MATTER_CONGRUENCE","K6",1.0,"Uses existing matter variables"),
    ("MATTER_CONGRUENCE","K7",1.0,"No circular dependence"),
    ("MATTER_CONGRUENCE","K8",1.0,"GR matter congruence exists"),
    ("MATTER_CONGRUENCE","K9",0.7,"Matter evolution law exists but not global GVH selection"),
    ("MATTER_CONGRUENCE","K10",0.5,"Environmental dependence is testable"),

    ("INDEPENDENT_VECTOR","K1",1.0,"Manifestly covariant"),
    ("INDEPENDENT_VECTOR","K2",1.0,"Constraint imposed by multiplier"),
    ("INDEPENDENT_VECTOR","K3",1.0,"Unique field after initial data are specified"),
    ("INDEPENDENT_VECTOR","K4",1.0,"Defined in vacuum"),
    ("INDEPENDENT_VECTOR","K5",1.0,"Independent of matter-fluid counting"),
    ("INDEPENDENT_VECTOR","K6",0.0,"Adds a new object"),
    ("INDEPENDENT_VECTOR","K7",1.0,"No circular dependence"),
    ("INDEPENDENT_VECTOR","K8",0.8,"Decoupling limit can recover GR"),
    ("INDEPENDENT_VECTOR","K9",0.0,"No GVH vector action selected"),
    ("INDEPENDENT_VECTOR","K10",1.0,"Preferred-frame and wave signatures are testable"),
]

evaluation_df = pd.DataFrame(
    rows,
    columns=[
        "origin_id",
        "criterion_id",
        "score",
        "justification",
    ],
).merge(
    criteria_df,
    on="criterion_id",
).merge(
    origins_df[["origin_id", "definition"]],
    on="origin_id",
)

evaluation_df

,origin_id,criterion_id,score,justification,criterion,hard_gate,definition
0,OBSERVER_CONGRUENCE,K1,1.0,Covariant congruence once observer family is specified,Covariant definition,True,u^mu is the observer congruence
1,OBSERVER_CONGRUENCE,K2,1.0,Can be normalized,Unit normalization derivable,True,u^mu is the observer congruence
2,OBSERVER_CONGRUENCE,K3,0.0,Observer family is not unique,Unique timelike direction,True,u^mu is the observer congruence
3,OBSERVER_CONGRUENCE,K4,1.0,Can be defined in vacuum,Defined in vacuum,True,u^mu is the observer congruence
4,OBSERVER_CONGRUENCE,K5,1.0,Independent of fluid multiplicity,Defined with multiple matter fluids,False,u^mu is the observer congruence
5,OBSERVER_CONGRUENCE,K6,0.7,Observer frame is tracked in GVH but not dynamically selected,Constructed from existing GVH objects,True,u^mu is the observer congruence
6,OBSERVER_CONGRUENCE,K7,1.0,No circular dependence,No circular dependence on Pi_mn[T],True,u^mu is the observer congruence
7,OBSERVER_CONGRUENCE,K8,1.0,GR observer congruences exist,GR limit identifiable,True,u^mu is the observer congruence
8,OBSERVER_CONGRUENCE,K9,0.0,No GVH evolution law selecting one congruence,Action or evolution law available,True,u^mu is the observer congruence
9,OBSERVER_CONGRUENCE,K10,0.3,Predictions are frame-dependent unless a selection rule exists,Falsifiable consequence identifiable,True,u^mu is the observer congruence


# 7. Portes dures

In [9]:
HARD_THRESHOLD = 0.75

evaluation_df["gate_pass"] = np.where(
    evaluation_df["hard_gate"],
    evaluation_df["score"] >= HARD_THRESHOLD,
    True,
)

hard_gate_summary_df = (
    evaluation_df.loc[
        evaluation_df["hard_gate"]
    ]
    .groupby("origin_id", as_index=False)
    .agg(
        hard_gates_passed=("gate_pass", "sum"),
        hard_gate_count=("gate_pass", "count"),
        all_hard_gates_pass=("gate_pass", "all"),
    )
)

hard_gate_summary_df

,origin_id,hard_gates_passed,hard_gate_count,all_hard_gates_pass
0,INDEPENDENT_VECTOR,7,9,False
1,MATTER_CONGRUENCE,5,9,False
2,OBSERVER_CONGRUENCE,5,9,False
3,SCALAR_CLOCK,7,9,False
4,TENSOR_EIGENVECTOR,4,9,False


# 8. Score descriptif non décisionnel

In [10]:
score_summary_df = (
    evaluation_df
    .groupby("origin_id", as_index=False)
    .agg(
        mean_score=("score", "mean"),
        minimum_score=("score", "min"),
    )
    .merge(
        hard_gate_summary_df,
        on="origin_id",
    )
    .sort_values(
        "mean_score",
        ascending=False,
    )
    .reset_index(drop=True)
)

score_summary_df["rank"] = (
    np.arange(len(score_summary_df)) + 1
)

score_summary_df

,origin_id,mean_score,minimum_score,hard_gates_passed,hard_gate_count,all_hard_gates_pass,rank
0,INDEPENDENT_VECTOR,0.78,0.0,7,9,False,1
1,SCALAR_CLOCK,0.76,0.0,7,9,False,2
2,TENSOR_EIGENVECTOR,0.72,0.0,4,9,False,3
3,OBSERVER_CONGRUENCE,0.70,0.0,5,9,False,4
4,MATTER_CONGRUENCE,0.67,0.0,5,9,False,5


Le score sert uniquement à repérer les constructions les moins incomplètes.

Il ne constitue pas une dérivation physique.

Une origine n’est acceptée que si toutes les portes dures passent et si l’action ou la loi d’évolution découle des fondements GVH.

# 9. Test du candidat vecteur propre tensoriel

In [11]:
lambda_eig = sp.symbols(
    "lambda_eig",
    real=True,
)

D00, D11, D22, D33 = sp.symbols(
    "D00 D11 D22 D33",
    real=True,
)

D_mixed = sp.diag(
    D00,
    D11,
    D22,
    D33,
)

eigenvalues = D_mixed.eigenvals()

tensor_eigenvector_test_df = pd.DataFrame([{
    "candidate_tensor": "diagonal D^mu_nu",
    "eigenvalues": str(eigenvalues),
    "timelike_eigenvector_possible": True,
    "unique_if_non_degenerate": True,
    "undefined_when_D_mn_zero_or_degenerate": True,
}])

tensor_eigenvector_test_df

,candidate_tensor,eigenvalues,timelike_eigenvector_possible,unique_if_non_degenerate,undefined_when_D_mn_zero_or_degenerate
0,diagonal D^mu_nu,"{D00: 1, D11: 1, D22: 1, D33: 1}",True,True,True


Le candidat tensoriel est intéressant parce qu’il pourrait relier directement \(u^\mu\) à la structure Diagonal Cubic.

On chercherait :

\[
D^\mu{}_\nu u^\nu
=
\lambda_D u^\mu,
\]

avec :

\[
u^\mu u_\mu=-1.
\]

Mais cette construction échoue comme dérivation globale actuelle lorsque :

- \(D_{\mu\nu}=0\) dans la limite GR ;
- plusieurs valeurs propres sont dégénérées ;
- aucune dynamique de \(D_{\mu\nu}\) ne garantit une valeur propre temporelle unique.

Il s’agit donc d’une piste GVH-native, mais pas encore d’une origine fermée.

# 10. Circularité avec le projecteur de source

In [12]:
circularity_df = pd.DataFrame([
    {
        "construction": "derive u^mu from Pi_mn[T]",
        "circular": True,
        "reason": (
            "Pi_mn[T] already requires h_mn and therefore u^mu"
        ),
    },
    {
        "construction": "derive u^mu from D_mn eigenvector",
        "circular": False,
        "reason": (
            "possible if D_mn exists independently of u^mu"
        ),
    },
    {
        "construction": "derive u^mu from scalar clock",
        "circular": False,
        "reason": "requires a new scalar sector",
    },
])

circularity_df

,construction,circular,reason
0,derive u^mu from Pi_mn[T],True,Pi_mn[T] already requires h_mn and therefore u^mu
1,derive u^mu from D_mn eigenvector,False,possible if D_mn exists independently of u^mu
2,derive u^mu from scalar clock,False,requires a new scalar sector


# 11. Lien avec le temps directionnel exploratoire

In [13]:
directional_time_link_df = pd.DataFrame([
    {
        "object": "tau_x, tau_y, tau_z",
        "current_status": "exploratory",
        "possible_role": (
            "components or projections of a temporal structure"
        ),
        "sufficient_to_define_u^mu": False,
    },
    {
        "object": "tau_G",
        "current_status": "exploratory",
        "possible_role": (
            "candidate global scalar clock"
        ),
        "sufficient_to_define_u^mu": False,
    },
    {
        "object": "gradient of tau_G",
        "current_status": "not derived",
        "possible_role": (
            "candidate normalized scalar-gradient origin"
        ),
        "sufficient_to_define_u^mu": (
            "only after tau_G becomes a covariant scalar field"
        ),
    },
])

directional_time_link_df

,object,current_status,possible_role,sufficient_to_define_u^mu
0,"tau_x, tau_y, tau_z",exploratory,components or projections of a temporal structure,False
1,tau_G,exploratory,candidate global scalar clock,False
2,gradient of tau_G,not derived,candidate normalized scalar-gradient origin,only after tau_G becomes a covariant scalar field


Le temps directionnel exploratoire pourrait fournir une origine au modèle scalaire :

\[
u_\mu
\propto
-\nabla_\mu\tau_G.
\]

Mais cela exige d’abord de démontrer que \(\tau_G\) est :

- un champ scalaire covariant ;
- défini indépendamment des coordonnées ;
- doté d’un gradient temporel partout dans le domaine ;
- gouverné par une dynamique propre.

Cette correspondance reste donc une hypothèse de recherche, pas un résultat acquis.

# 12. Exigences pour une dérivation GVH-native

In [14]:
native_derivation_requirements_df = pd.DataFrame([
    {
        "requirement": (
            "Existing GVH object defines a timelike direction"
        ),
        "satisfied": False,
    },
    {
        "requirement": (
            "Direction remains unique in vacuum"
        ),
        "satisfied": False,
    },
    {
        "requirement": (
            "GR limit remains regular"
        ),
        "satisfied": False,
    },
    {
        "requirement": (
            "Evolution equation follows from GVH action"
        ),
        "satisfied": False,
    },
    {
        "requirement": (
            "At least one observable differs from GR"
        ),
        "satisfied": False,
    },
])

native_derivation_requirements_df

,requirement,satisfied
0,Existing GVH object defines a timelike direction,False
1,Direction remains unique in vacuum,False
2,GR limit remains regular,False
3,Evolution equation follows from GVH action,False
4,At least one observable differs from GR,False


# 13. Postulat minimal candidat — non adopté

Le candidat le plus directement lié aux idées de temps directionnel serait :

\[
\boxed{
u_\mu
=
-\frac{\nabla_\mu\tau_G}
{\sqrt{-\nabla_\alpha\tau_G\nabla^\alpha\tau_G}}
}
\]

à condition que :

\[
\nabla_\alpha\tau_G\nabla^\alpha\tau_G<0.
\]

Ce principe sélectionnerait le modèle `SCALAR_CLOCK`.

Mais il ajouterait un nouveau contenu physique : \(\tau_G\) deviendrait un champ scalaire fondamental ou effectif.

Ce notebook enregistre ce candidat sans l’adopter.

In [15]:
candidate_postulate_df = pd.DataFrame([{
    "postulate_id": "P-GVH-TIME-1",
    "statement": (
        "The GVH global directional time tau_G is a covariant "
        "scalar field with timelike gradient, and u_mu is its "
        "normalized negative gradient."
    ),
    "selected_model": "SCALAR_CLOCK",
    "accepted": False,
    "reason_not_accepted": (
        "tau_G has not yet been established as a covariant "
        "dynamical scalar field"
    ),
    "required_prediction": (
        "derive a scalar action and a nontrivial observable "
        "before observational fitting"
    ),
}])

candidate_postulate_df

,postulate_id,statement,selected_model,accepted,reason_not_accepted,required_prediction
0,P-GVH-TIME-1,"The GVH global directional time tau_G is a covariant scalar field with timelike gradient, and u_mu is its normalized negative gradient.",SCALAR_CLOCK,False,tau_G has not yet been established as a covariant dynamical scalar field,derive a scalar action and a nontrivial observable before observational fitting


# 14. Falsifiabilité du postulat candidat

In [16]:
falsifiability_df = pd.DataFrame([
    {
        "test": "hypersurface orthogonality",
        "prediction": "omega_mn = 0",
        "status": "DERIVABLE IF POSTULATE ADOPTED",
    },
    {
        "test": "preferred foliation",
        "prediction": (
            "possible frame/clock signatures"
        ),
        "status": "ACTION REQUIRED",
    },
    {
        "test": "weak-field metric",
        "prediction": (
            "alpha_t and alpha_s become functions "
            "of scalar-sector coefficients"
        ),
        "status": "ACTION REQUIRED",
    },
    {
        "test": "GR limit",
        "prediction": (
            "tau_G constant or scalar sector decouples"
        ),
        "status": "REQUIRED",
    },
])

falsifiability_df

,test,prediction,status
0,hypersurface orthogonality,omega_mn = 0,DERIVABLE IF POSTULATE ADOPTED
1,preferred foliation,possible frame/clock signatures,ACTION REQUIRED
2,weak-field metric,alpha_t and alpha_s become functions of scalar-sector coefficients,ACTION REQUIRED
3,GR limit,tau_G constant or scalar sector decouples,REQUIRED


# 15. Variables de décision scientifique

In [17]:
candidate_origins_identified = bool(
    len(origins_df) >= 3
)

hard_gate_origins = (
    hard_gate_summary_df.loc[
        hard_gate_summary_df[
            "all_hard_gates_pass"
        ],
        "origin_id",
    ].tolist()
)

dynamics_available = {
    "OBSERVER_CONGRUENCE": False,
    "SCALAR_CLOCK": False,
    "TENSOR_EIGENVECTOR": False,
    "MATTER_CONGRUENCE": False,
    "INDEPENDENT_VECTOR": False,
}

physically_derived_origins = [
    origin
    for origin in hard_gate_origins
    if dynamics_available.get(
        origin,
        False,
    )
]

unique_physical_origin = bool(
    len(physically_derived_origins) == 1
)

NEW_PHYSICAL_POSTULATE_REQUIRED = bool(
    not unique_physical_origin
)

if unique_physical_origin:
    FINAL_STATUS = (
        "PASS-TIMELIKE-FIELD-DERIVED-FROM-GVH-GEOMETRY"
    )
elif candidate_origins_identified:
    FINAL_STATUS = (
        "PASS-CANDIDATE-TIMELIKE-ORIGINS-IDENTIFIED_"
        "BLOCKED-UNIQUE-PHYSICAL-ORIGIN_"
        "NEW-PHYSICAL-POSTULATE-REQUIRED"
    )
else:
    FINAL_STATUS = (
        "BLOCKED-TIMELIKE-ORIGIN-CLASSIFICATION"
    )

decision_state_df = pd.DataFrame([{
    "candidate_origins_identified": (
        candidate_origins_identified
    ),
    "hard_gate_origins": ",".join(
        hard_gate_origins
    ),
    "physically_derived_origins": ",".join(
        physically_derived_origins
    ),
    "unique_physical_origin": (
        unique_physical_origin
    ),
    "new_physical_postulate_required": (
        NEW_PHYSICAL_POSTULATE_REQUIRED
    ),
    "final_status": FINAL_STATUS,
}])

decision_state_df

,candidate_origins_identified,hard_gate_origins,physically_derived_origins,unique_physical_origin,new_physical_postulate_required,final_status
0,True,,,False,True,PASS-CANDIDATE-TIMELIKE-ORIGINS-IDENTIFIED_BLOCKED-UNIQUE-PHYSICAL-ORIGIN_NEW-PHYSICAL-POSTULATE-REQUIRED


# 16. Artefact de gouvernance

In [18]:
origin_artifact = {
    "artifact_status": (
        "UNIQUE_ORIGIN_DERIVED"
        if unique_physical_origin
        else (
            "CANDIDATE_ORIGINS_IDENTIFIED_"
            "UNIQUE_ORIGIN_BLOCKED"
        )
    ),
    "candidate_origins": origins_df[
        "origin_id"
    ].tolist(),
    "physically_derived_origin": (
        physically_derived_origins[0]
        if unique_physical_origin
        else None
    ),
    "leading_GVH_native_candidate": (
        "normalized timelike eigenvector of D^mu_nu"
    ),
    "leading_directional_time_candidate": (
        "normalized gradient of covariant tau_G"
    ),
    "candidate_postulate": {
        "id": "P-GVH-TIME-1",
        "accepted": False,
        "formula": (
            "u_mu = -grad_mu(tau_G)/"
            "sqrt(-grad(tau_G)^2)"
        ),
    },
    "blocking_points": [
        (
            "tau_G is not yet a covariant "
            "dynamical scalar"
        ),
        (
            "D_mn does not yet guarantee a "
            "unique timelike eigenvector"
        ),
        (
            "no candidate has a complete "
            "GVH-derived action"
        ),
        (
            "regularity of the GR limit "
            "remains unresolved"
        ),
    ],
    "source_notebooks": [
        "0.2.21",
        "0.2.22",
        "0.2.23",
        "0.2.23.1",
        "0.2.23.2",
        "0.2.23.3.1",
        "0.2.23.4.1",
    ],
    "source_traceability_pass": (
        source_traceability_pass
    ),
}

origin_artifact

{'artifact_status': 'CANDIDATE_ORIGINS_IDENTIFIED_UNIQUE_ORIGIN_BLOCKED',
 'candidate_origins': ['OBSERVER_CONGRUENCE',
  'SCALAR_CLOCK',
  'TENSOR_EIGENVECTOR',
  'MATTER_CONGRUENCE',
  'INDEPENDENT_VECTOR'],
 'physically_derived_origin': None,
 'leading_GVH_native_candidate': 'normalized timelike eigenvector of D^mu_nu',
 'leading_directional_time_candidate': 'normalized gradient of covariant tau_G',
 'candidate_postulate': {'id': 'P-GVH-TIME-1',
  'accepted': False,
  'formula': 'u_mu = -grad_mu(tau_G)/sqrt(-grad(tau_G)^2)'},
 'blocking_points': ['tau_G is not yet a covariant dynamical scalar',
  'D_mn does not yet guarantee a unique timelike eigenvector',
  'no candidate has a complete GVH-derived action',
  'regularity of the GR limit remains unresolved'],
 'source_notebooks': ['0.2.21',
  '0.2.22',
  '0.2.23',
  '0.2.23.1',
  '0.2.23.2',
  '0.2.23.3.1',
  '0.2.23.4.1'],
 'source_traceability_pass': True}

# 17. Contrôle bloquant de traçabilité de l’artefact

In [19]:
CANONICAL_TIMELIKE_DEPENDENCIES = [
    "0.2.23.3.1",
    "0.2.23.4.1",
]

DEPRECATED_TIMELIKE_DEPENDENCIES = [
    "0.2.23.3",
    "0.2.23.4",
]

artifact_source_notebooks = list(
    origin_artifact.get(
        "source_notebooks",
        [],
    )
)

missing_canonical_dependencies = [
    dependency
    for dependency in CANONICAL_TIMELIKE_DEPENDENCIES
    if dependency not in artifact_source_notebooks
]

deprecated_dependencies_present = [
    dependency
    for dependency in DEPRECATED_TIMELIKE_DEPENDENCIES
    if dependency in artifact_source_notebooks
]

artifact_traceability_pass = bool(
    source_traceability_pass
    and dependency_02331_pass
    and dependency_02341_pass
    and not missing_canonical_dependencies
    and not deprecated_dependencies_present
)

assert artifact_traceability_pass, (
    "TRACEABILITY ASSERT FAILED — "
    f"source_traceability_pass="
    f"{source_traceability_pass}; "
    f"dependency_02331_pass="
    f"{dependency_02331_pass}; "
    f"dependency_02341_pass="
    f"{dependency_02341_pass}; "
    f"missing canonical dependencies="
    f"{missing_canonical_dependencies}; "
    f"deprecated dependencies present="
    f"{deprecated_dependencies_present}; "
    f"artifact source_notebooks="
    f"{artifact_source_notebooks}"
)

print("Artefact traceability assert : PASS")
print(
    "source_notebooks :",
    artifact_source_notebooks,
)

Artefact traceability assert : PASS
source_notebooks : ['0.2.21', '0.2.22', '0.2.23', '0.2.23.1', '0.2.23.2', '0.2.23.3.1', '0.2.23.4.1']


# 18. Décision scientifique finale

In [20]:
decision_df = pd.DataFrame([{
    "final_status": FINAL_STATUS,
    "candidate_origins_identified": (
        candidate_origins_identified
    ),
    "hard_gate_origins": ",".join(
        hard_gate_origins
    ),
    "physically_derived_origins": ",".join(
        physically_derived_origins
    ),
    "unique_physical_origin": (
        unique_physical_origin
    ),
    "artifact_traceability_pass": (
        artifact_traceability_pass
    ),
    "new_physical_postulate_required": (
        NEW_PHYSICAL_POSTULATE_REQUIRED
    ),
    "leading_GVH_native_candidate": (
        "TENSOR_EIGENVECTOR"
    ),
    "leading_directional_time_candidate": (
        "SCALAR_CLOCK via tau_G"
    ),
}])

print("STATUT FINAL :", FINAL_STATUS)
decision_df

STATUT FINAL : PASS-CANDIDATE-TIMELIKE-ORIGINS-IDENTIFIED_BLOCKED-UNIQUE-PHYSICAL-ORIGIN_NEW-PHYSICAL-POSTULATE-REQUIRED


,final_status,candidate_origins_identified,hard_gate_origins,physically_derived_origins,unique_physical_origin,artifact_traceability_pass,new_physical_postulate_required,leading_GVH_native_candidate,leading_directional_time_candidate
0,PASS-CANDIDATE-TIMELIKE-ORIGINS-IDENTIFIED_BLOCKED-UNIQUE-PHYSICAL-ORIGIN_NEW-PHYSICAL-POSTULATE-REQUIRED,True,,,False,True,True,TENSOR_EIGENVECTOR,SCALAR_CLOCK via tau_G


# 19. Exports

In [21]:
PREFIX = "GVH_Diagonal_Cubic_0.2.23.5.1"

exports = {
    "Source_Notebooks": source_notebooks_df,
    "Critical_Dependencies": (
        critical_dependencies_df
    ),
    "Existing_Objects": existing_objects_df,
    "Origins": origins_df,
    "Criteria": criteria_df,
    "Evaluation": evaluation_df,
    "Hard_Gate_Summary": (
        hard_gate_summary_df
    ),
    "Score_Summary": score_summary_df,
    "Tensor_Eigenvector_Test": (
        tensor_eigenvector_test_df
    ),
    "Circularity": circularity_df,
    "Directional_Time_Link": (
        directional_time_link_df
    ),
    "Native_Derivation_Requirements": (
        native_derivation_requirements_df
    ),
    "Candidate_Postulate": (
        candidate_postulate_df
    ),
    "Falsifiability": falsifiability_df,
    "Decision_State": decision_state_df,
    "Decision": decision_df,
}

for suffix, table in exports.items():
    table.to_csv(
        EXPORT_DIR
        / f"{PREFIX}_{suffix}.csv",
        index=False,
    )

ORIGIN_FILE = (
    PROCESSED_DIR
    / "gvh_timelike_field_physical_origin.json"
)

ORIGIN_FILE.write_text(
    json.dumps(
        origin_artifact,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

metadata = {
    "notebook": (
        "GVH_Diagonal_Cubic_0.2.23.5.1_"
        "Timelike_Field_Physical_Postulate_and_Origin"
    ),
    "execution_time_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "final_status": FINAL_STATUS,
    "all_sources_found": (
        all_sources_found
    ),
    "all_sources_canonical": (
        all_sources_canonical
    ),
    "source_traceability_pass": (
        source_traceability_pass
    ),
    "dependency_0.2.23.3.1_pass": (
        dependency_02331_pass
    ),
    "dependency_0.2.23.4.1_pass": (
        dependency_02341_pass
    ),
    "artifact_traceability_pass": (
        artifact_traceability_pass
    ),
    "candidate_origins_identified": (
        candidate_origins_identified
    ),
    "unique_physical_origin": (
        unique_physical_origin
    ),
    "new_physical_postulate_required": (
        NEW_PHYSICAL_POSTULATE_REQUIRED
    ),
    "origin_file": str(
        ORIGIN_FILE
    ),
    "next_action": (
        "Audit P-GVH-TIME-1 as an exploratory postulate, "
        "or derive a unique timelike eigenvector condition "
        "from the D_mn dynamics."
    ),
}

with (
    EXPORT_DIR
    / f"{PREFIX}_Metadata.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
        ensure_ascii=False,
    )

print("Origin artifact :", ORIGIN_FILE)
print("Exports :", EXPORT_DIR)

Origin artifact : /content/Univers/gvh_diagonal_cubic/data/processed/timelike_field/gvh_timelike_field_physical_origin.json
Exports : /content/Univers/gvh_diagonal_cubic/exports


# Conclusion

Le notebook identifie plusieurs origines possibles pour \(u^\mu\), mais aucune n’est actuellement dérivée de façon complète et unique par GVH.

Deux pistes restent particulièrement importantes.

## Piste tensorielle GVH-native

\[
D^\mu{}_\nu u^\nu
=
\lambda_Du^\mu.
\]

Elle n’ajoute pas nécessairement de nouveau champ, mais exige une dynamique de \(D_{\mu\nu}\) garantissant une valeur propre temporelle unique et une limite GR régulière.

## Piste du temps directionnel global

\[
u_\mu
=
-\frac{\nabla_\mu\tau_G}
{\sqrt{-\nabla_\alpha\tau_G\nabla^\alpha\tau_G}}.
\]

Elle pourrait relier le secteur covariant au programme de temps directionnel, mais seulement si \(\tau_G\) devient un champ scalaire covariant et dynamique.

Le statut attendu est donc :

```text
PASS-CANDIDATE-TIMELIKE-ORIGINS-IDENTIFIED
BLOCKED-UNIQUE-PHYSICAL-ORIGIN
NEW-PHYSICAL-POSTULATE-REQUIRED
```

Aucun postulat nouveau n’est adopté automatiquement dans ce notebook.